In [76]:
# !pip install pyreadr

In [95]:
all_radiomics_features=pd.read_csv('all_radiomics_features.csv')
all_radiomics_cases = all_radiomics_features['Case_ID'].to_list()
imaging_cases = []
for case in all_radiomics_cases:
    # print(case)
    corrected_case = case.replace('MR_EGD','EGD').replace('UCSF-PDGM-0','UCSF-PDGM-').replace('_nifti','')
    imaging_cases.append(corrected_case)
for i in imaging_cases:
    # print(i)
    pass

In [96]:
import pandas as pd
import pyreadr

monet_rdata = 'Integrated_data_2021.RData'   # This is from Mendoca work MONET repo: Scripts-for-reproducibility
require_grade4_for_gbm = True

In [97]:
digest_lgg = pd.read_excel('doiJNLP-JAMS4RFq-nbia-digest-1 (2).xlsx')
digest_gbm = pd.read_excel('doiJNLP-QoOaKUdn-nbia-digest-1 (4).xlsx')
digest = pd.concat([digest_lgg, digest_gbm], ignore_index=True)
digest = digest[(digest['Modality'] == 'MR') & (digest['Manufacturer'].notna())]

scanner_by_patient = {}
for pid, manuf in zip(digest['Patient ID'], digest['Manufacturer'].astype(str).str.upper()):
    if 'GE' in manuf:
        scanner_by_patient[pid] = 'GE'
    elif 'SIEMENS' in manuf:
        scanner_by_patient[pid] = 'Siemens'
    elif 'PHILIPS' in manuf:
        scanner_by_patient[pid] = 'Philips'
    else:
        scanner_by_patient[pid] = 'Other'



In [79]:
monet = pyreadr.read_r(monet_rdata)
monet

OrderedDict([('astro_clinic',
                              Patient_ID years_to_birth vital_status days_to_death  \
              rownames                                                               
              TCGA-02-0010  TCGA-02-0010             20            1          1077   
              TCGA-02-0014  TCGA-02-0014             25            1          2512   
              TCGA-02-0026  TCGA-02-0026             27            1           748   
              TCGA-02-0058  TCGA-02-0058             28            1           254   
              TCGA-02-0080  TCGA-02-0080             28            1          2729   
              ...                    ...            ...          ...           ...   
              TCGA-WY-A85A  TCGA-WY-A85A             20            0           NaN   
              TCGA-WY-A85B  TCGA-WY-A85B             24            0           NaN   
              TCGA-WY-A85C  TCGA-WY-A85C             36            0           NaN   
              TCGA-WY-A8

In [98]:
astro_df = monet['astro_clinic'].copy()
oligo_df = monet['oligo_clinic'].copy()
gbm_df = monet['gbm_clinic2021'].copy()
astro_df['glioma_class'] = 'astrocytoma'
oligo_df['glioma_class'] = 'oligodendroglioma'
gbm_df['glioma_class'] = 'glioblastoma'
tcga = pd.concat([astro_df, oligo_df, gbm_df], ignore_index=True)

tcga_out = pd.DataFrame()
tcga_out['case_id'] = tcga['Patient_ID']
tcga_out['site'] = 'TCGA'
tcga_out['glioma_class'] = tcga['glioma_class']

tcga_idh = []
tcga_codel = []
for cls in tcga['glioma_class']:
    if cls == 'oligodendroglioma':      # IDHmut-codel
        tcga_idh.append(1)
        tcga_codel.append(1)
    elif cls == 'astrocytoma':          # IDHmut-non-codel
        tcga_idh.append(1)
        tcga_codel.append(0)
    else:                               # glioblastoma = IDHwt
        tcga_idh.append(0)
        tcga_codel.append(0)
tcga_out['idh'] = tcga_idh
tcga_out['codel_1p19q'] = tcga_codel

tcga_out['age'] = pd.to_numeric(tcga['years_to_birth'], errors='coerce').fillna(-1)

tcga_sex = []
for v in tcga['gender'].astype(str).str.strip().str.lower():
    if v in ('m', 'male'):
        tcga_sex.append(1)
    elif v in ('f', 'female'):
        tcga_sex.append(0)
    else:
        tcga_sex.append(-1)
tcga_out['sex'] = tcga_sex

tcga_scanner = []
for pid in tcga['Patient_ID']:
    if pid in scanner_by_patient:
        tcga_scanner.append(scanner_by_patient[pid])
    else:
        tcga_scanner.append('Unknown')
tcga_out['Scanner'] = tcga_scanner

tcga_out['field_strength'] = -1   # not available for TCGA

tcga_out = tcga_out[tcga_out['case_id'].isin(imaging_cases)]
print(tcga_out.shape)

tcga_out['glioma_class'].value_counts()

(140, 9)


glioma_class
glioblastoma         84
astrocytoma          43
oligodendroglioma    13
Name: count, dtype: int64

In [99]:
ucsf = pd.read_csv('UCSF-PDGM-metadata_v5 (1).csv')

ucsf_out = pd.DataFrame()
ucsf_out['case_id'] = ucsf['ID']
ucsf_out['site'] = 'UCSF'

ucsf_out['glioma_class'] = ''
dx = ucsf['Final pathologic diagnosis (WHO 2021)']
ucsf_out.loc[dx == 'Astrocytoma, IDH-mutant', 'glioma_class'] = 'astrocytoma'
ucsf_out.loc[dx == 'Oligodendroglioma, IDH-mutant, 1p/19q-codeleted', 'glioma_class'] = 'oligodendroglioma'
ucsf_out.loc[dx == 'Glioblastoma, IDH-wildtype', 'glioma_class'] = 'glioblastoma'

ucsf_idh = []
for v in ucsf['IDH'].astype(str).str.strip().str.lower():
    if v == 'wildtype':
        ucsf_idh.append(0)
    elif v in ('nan', 'unknown', ''):
        ucsf_idh.append(-1)
    else:
        ucsf_idh.append(1)            # any listed IDH mutation
ucsf_out['idh'] = ucsf_idh

ucsf_codel = []
for v in ucsf['1p/19q'].astype(str).str.strip().str.lower():
    if 'co-deletion' in v:            # 'co-deletion' and 'relative co-deletion'
        ucsf_codel.append(1)
    elif v == 'intact':
        ucsf_codel.append(0)
    else:
        ucsf_codel.append(-1)
ucsf_out['codel_1p19q'] = ucsf_codel

ucsf_out['age'] = pd.to_numeric(ucsf['Age at MRI'], errors='coerce').fillna(-1)

ucsf_sex = []
for v in ucsf['Sex'].astype(str).str.strip().str.lower():
    if v in ('m', 'male'):
        ucsf_sex.append(1)
    elif v in ('f', 'female'):
        ucsf_sex.append(0)
    else:
        ucsf_sex.append(-1)
ucsf_out['sex'] = ucsf_sex

ucsf_out['Scanner'] = 'GE'
ucsf_out['field_strength'] = 3.0

ucsf_out = ucsf_out[ucsf_out['glioma_class'].isin(['astrocytoma', 'oligodendroglioma', 'glioblastoma'])]

ucsf_out = ucsf_out[ucsf_out['case_id'].isin(imaging_cases)]
print(ucsf_out.shape)

print(ucsf_out.shape)
ucsf_out['glioma_class'].value_counts()


(471, 9)
(471, 9)


glioma_class
glioblastoma         368
astrocytoma           90
oligodendroglioma     13
Name: count, dtype: int64

In [100]:
egd = pd.read_excel('EGD_Clinical_data.xlsx')

egd_out = pd.DataFrame()
egd_out['case_id'] = egd['Subject']
egd_out['site'] = 'EGD'

egd_out['glioma_class'] = ''
egd_out.loc[(egd['IDH'] == 1) & (egd['1p19q'] == 1), 'glioma_class'] = 'oligodendroglioma'
egd_out.loc[(egd['IDH'] == 1) & (egd['1p19q'] == 0), 'glioma_class'] = 'astrocytoma'
if require_grade4_for_gbm == False:
    egd_out.loc[(egd['IDH'] == 0), 'glioma_class'] = 'glioblastoma'
if require_grade4_for_gbm == True:
    egd_out.loc[(egd['IDH'] == 0) & (egd['Grade'] == 4), 'glioma_class'] = 'glioblastoma'

egd_out['idh'] = egd['IDH']                 # already 1=mutant / 0=wildtype / -1=unknown
egd_out['codel_1p19q'] = egd['1p19q']       # already 1=codel / 0=non-codel / -1=unknown

egd_out['age'] = pd.to_numeric(egd['Age'], errors='coerce').fillna(-1)

egd_sex = []
for v in egd['Sex'].astype(str).str.strip().str.lower():
    if v in ('m', 'male'):
        egd_sex.append(1)
    elif v in ('f', 'female'):
        egd_sex.append(0)
    else:
        egd_sex.append(-1)
egd_out['sex'] = egd_sex

egd_scanner = []
for v in egd['Manufacturer'].astype(str).str.upper():
    if 'GE' in v:
        egd_scanner.append('GE')
    elif 'SIEMENS' in v:
        egd_scanner.append('Siemens')
    elif 'PHILIPS' in v:
        egd_scanner.append('Philips')
    else:
        egd_scanner.append('Other')
egd_out['Scanner'] = egd_scanner

egd_out['field_strength'] = pd.to_numeric(egd['Field'], errors='coerce').fillna(-1)

egd_out = egd_out[egd_out['glioma_class'].isin(['astrocytoma', 'oligodendroglioma', 'glioblastoma'])]

egd_out = egd_out[egd_out['case_id'].isin(imaging_cases)]
print(egd_out.shape)

print(egd_out.shape)
egd_out['glioma_class'].value_counts()


(404, 9)
(404, 9)


glioma_class
glioblastoma         254
astrocytoma           77
oligodendroglioma     73
Name: count, dtype: int64

In [101]:
utsw = pd.read_csv('UTSW_Glioma_Metadata-2-1.tsv', sep='\t')

utsw_out = pd.DataFrame()
utsw_out['case_id'] = utsw['Subject ID']
utsw_out['site'] = 'UTSW'

utsw_out['glioma_class'] = ''
utsw_out.loc[(utsw['IDH'] == 'mutated') & (utsw['1p19Q CODEL'] == 'co-deleted'), 'glioma_class'] = 'oligodendroglioma'
utsw_out.loc[(utsw['IDH'] == 'mutated') & (utsw['1p19Q CODEL'] == 'non co-deleted'), 'glioma_class'] = 'astrocytoma'
if require_grade4_for_gbm == False:
    utsw_out.loc[(utsw['IDH'] == 'wild type'), 'glioma_class'] = 'glioblastoma'
if require_grade4_for_gbm == True:
    utsw_out.loc[(utsw['IDH'] == 'wild type') & (utsw['Tumor Grade'] == 4), 'glioma_class'] = 'glioblastoma'

utsw_idh = []
for v in utsw['IDH'].astype(str).str.strip().str.lower():
    if v == 'mutated':
        utsw_idh.append(1)
    elif v == 'wild type':
        utsw_idh.append(0)
    else:
        utsw_idh.append(-1)
utsw_out['idh'] = utsw_idh

utsw_codel = []
for v in utsw['1p19Q CODEL'].astype(str).str.strip().str.lower():
    if v == 'co-deleted':
        utsw_codel.append(1)
    elif v == 'non co-deleted':
        utsw_codel.append(0)
    else:
        utsw_codel.append(-1)
utsw_out['codel_1p19q'] = utsw_codel

utsw_out['age'] = pd.to_numeric(utsw['Age at Imaging'], errors='coerce').fillna(-1)

utsw_sex = []
for v in utsw['Sex at birth'].astype(str).str.strip().str.lower():
    if v in ('m', 'male'):
        utsw_sex.append(1)
    elif v in ('f', 'female'):
        utsw_sex.append(0)
    else:
        utsw_sex.append(-1)
utsw_out['sex'] = utsw_sex

utsw_scanner = []
for v in utsw['Scanner Make'].astype(str).str.upper():
    if 'GE' in v:
        utsw_scanner.append('GE')
    elif 'SIEMENS' in v:
        utsw_scanner.append('Siemens')
    elif 'PHILIPS' in v:
        utsw_scanner.append('Philips')
    else:
        utsw_scanner.append('Other')
utsw_out['Scanner'] = utsw_scanner

utsw_out['field_strength'] = pd.to_numeric(utsw['Scanner Strength'], errors='coerce').fillna(-1)

utsw_out = utsw_out[utsw_out['glioma_class'].isin(['astrocytoma', 'oligodendroglioma', 'glioblastoma'])]

utsw_out = utsw_out[utsw_out['case_id'].isin(imaging_cases)]
print(utsw_out.shape)

print(utsw_out.shape)
utsw_out['glioma_class'].value_counts()


(529, 9)
(529, 9)


glioma_class
glioblastoma         389
astrocytoma           77
oligodendroglioma     63
Name: count, dtype: int64

In [102]:
cols = ['case_id', 'site', 'glioma_class', 'idh', 'codel_1p19q', 'age', 'sex', 'Scanner', 'field_strength']
combined = pd.concat([tcga_out[cols], ucsf_out[cols], egd_out[cols], utsw_out[cols]], ignore_index=True)
combined.to_csv('combined_glioma_who2021.csv', index=False)

print(combined['site'].value_counts())
print('TOTAL:', len(combined))
print(combined.shape)
combined['glioma_class'].value_counts()


site
UTSW    529
UCSF    471
EGD     404
TCGA    140
Name: count, dtype: int64
TOTAL: 1544
(1544, 9)


glioma_class
glioblastoma         1095
astrocytoma           287
oligodendroglioma     162
Name: count, dtype: int64

In [103]:
# Build table 1 for manuscript 

import pandas as pd

combined = pd.read_csv('combined_glioma_who2021.csv')
cohorts = ['TCGA', 'UCSF', 'EGD', 'UTSW']

row_labels = []
columns = {}

for cohort in cohorts:
    d = combined[combined['site'] == cohort]
    n = len(d)
    cells = []
    labels = []

    # ---- Age ----
    age = d['age'][d['age'] != -1]
    labels.append('Age (years), median [IQR]')
    cells.append(f"{age.median():.1f} [{age.quantile(0.25):.1f}-{age.quantile(0.75):.1f}]")
    labels.append('Age (years), mean +/- SD')
    cells.append(f"{age.mean():.1f} +/- {age.std():.1f}")

    # ---- Sex ----
    male = (d['sex'] == 1).sum()
    female = (d['sex'] == 0).sum()
    sex_unknown = (d['sex'] == -1).sum()
    labels.append('Male')
    cells.append(f"{male} ({100*male/n:.1f}%)")
    labels.append('Female')
    cells.append(f"{female} ({100*female/n:.1f}%)")
    labels.append('Sex unknown')
    cells.append(f"{sex_unknown} ({100*sex_unknown/n:.1f}%)")

    # ---- Tumor classification (WHO 2021) ----
    gbm = (d['glioma_class'] == 'glioblastoma').sum()
    astro = (d['glioma_class'] == 'astrocytoma').sum()
    oligo = (d['glioma_class'] == 'oligodendroglioma').sum()
    labels.append('Glioblastoma')
    cells.append(f"{gbm} ({100*gbm/n:.1f}%)")
    labels.append('Astrocytoma')
    cells.append(f"{astro} ({100*astro/n:.1f}%)")
    labels.append('Oligodendroglioma')
    cells.append(f"{oligo} ({100*oligo/n:.1f}%)")

    # ---- IDH status ----
    idh_wt = (d['idh'] == 0).sum()
    idh_mut = (d['idh'] == 1).sum()
    idh_unknown = (d['idh'] == -1).sum()
    labels.append('IDH wildtype')
    cells.append(f"{idh_wt} ({100*idh_wt/n:.1f}%)")
    labels.append('IDH mutant')
    cells.append(f"{idh_mut} ({100*idh_mut/n:.1f}%)")
    labels.append('IDH unknown')
    cells.append(f"{idh_unknown} ({100*idh_unknown/n:.1f}%)")

    # ---- 1p/19q codeletion ----
    codel = (d['codel_1p19q'] == 1).sum()
    noncodel = (d['codel_1p19q'] == 0).sum()
    codel_unknown = (d['codel_1p19q'] == -1).sum()
    labels.append('1p/19q codeleted')
    cells.append(f"{codel} ({100*codel/n:.1f}%)")
    labels.append('1p/19q non-codeleted')
    cells.append(f"{noncodel} ({100*noncodel/n:.1f}%)")
    labels.append('1p/19q unknown/missing')
    cells.append(f"{codel_unknown} ({100*codel_unknown/n:.1f}%)")

    # ---- Scanner manufacturer ----
    ge = (d['Scanner'] == 'GE').sum()
    siemens = (d['Scanner'] == 'Siemens').sum()
    philips = (d['Scanner'] == 'Philips').sum()
    scanner_other = n - ge - siemens - philips      # 'Other' + 'Unknown'
    labels.append('Scanner GE')
    cells.append(f"{ge} ({100*ge/n:.1f}%)")
    labels.append('Scanner Siemens')
    cells.append(f"{siemens} ({100*siemens/n:.1f}%)")
    labels.append('Scanner Philips')
    cells.append(f"{philips} ({100*philips/n:.1f}%)")
    labels.append('Scanner Other/Unknown')
    cells.append(f"{scanner_other} ({100*scanner_other/n:.1f}%)")

    # ---- Magnetic field strength ----
    f30 = (d['field_strength'] == 3.0).sum()
    f15 = (d['field_strength'] == 1.5).sum()
    field_other = n - f30 - f15                     # other values + missing (-1)
    labels.append('Field 3.0T')
    cells.append(f"{f30} ({100*f30/n:.1f}%)")
    labels.append('Field 1.5T')
    cells.append(f"{f15} ({100*f15/n:.1f}%)")
    labels.append('Field Other/Unknown')
    cells.append(f"{field_other} ({100*field_other/n:.1f}%)")

    if cohort == cohorts[0]:
        row_labels = labels
    columns[f"{cohort} (N={n})"] = cells

table1 = pd.DataFrame({'Characteristic': row_labels})
for col in columns:
    table1[col] = columns[col]

table1.to_csv('table1_baseline.csv', index=False)
print(table1.to_string(index=False))

           Characteristic     TCGA (N=140)     UCSF (N=471)      EGD (N=404)     UTSW (N=529)
Age (years), median [IQR] 54.0 [41.0-62.0] 59.0 [46.0-68.0] 57.0 [45.0-69.0] 59.0 [45.0-68.0]
 Age (years), mean +/- SD    51.6 +/- 15.4    56.8 +/- 15.3    55.8 +/- 15.3    56.1 +/- 15.4
                     Male       74 (52.9%)      282 (59.9%)      246 (60.9%)      315 (59.5%)
                   Female       66 (47.1%)      189 (40.1%)      157 (38.9%)      214 (40.5%)
              Sex unknown         0 (0.0%)         0 (0.0%)         1 (0.2%)         0 (0.0%)
             Glioblastoma       84 (60.0%)      368 (78.1%)      254 (62.9%)      389 (73.5%)
              Astrocytoma       43 (30.7%)       90 (19.1%)       77 (19.1%)       77 (14.6%)
        Oligodendroglioma        13 (9.3%)        13 (2.8%)       73 (18.1%)       63 (11.9%)
             IDH wildtype       84 (60.0%)      368 (78.1%)      254 (62.9%)      389 (73.5%)
               IDH mutant       56 (40.0%)      103 (21.9%) 